## **Exo 1**

In [7]:
from __future__ import annotations
import bisect
import heapq
import logging
import math
import statistics
import subprocess
import sys

logger = logging.getLogger(__name__)

def stdlib_pack(values: list[float], k: int, x: float, probe: bool = False) -> dict[str, object]:
    clean = [v for v in values if math.isfinite(v)]
    filtered_out = len(values) - len(clean)
    logger.info("filtered_out=%d", filtered_out)

    if not clean:
        raise ValueError("Filtered list is empty")

    mean = statistics.mean(clean)
    median = statistics.median(clean)

    if k <= 0:
        k_smallest = []
    elif k > len(clean):
        k_smallest = sorted(clean)
    else:
        k_smallest = heapq.nsmallest(k, clean)

    sorted_clean = sorted(clean)
    idx = bisect.bisect_left(sorted_clean, x)
    with_x = sorted_clean[:]
    with_x.insert(idx, x)

    if probe:
        r = subprocess.run([sys.executable, "-c", "print(123)"], capture_output=True, text=True)
        probe_out = r.stdout.strip()
    else:
        probe_out = None

    return {
        "clean": clean,
        "mean": mean,
        "median": median,
        "k_smallest": k_smallest,
        "with_x": with_x,
        "probe_out": probe_out,
    }


# Tests
out = stdlib_pack([1.0, float("nan"), 2.0, float("inf"), -1.0], k=2, x=1.5, probe=False)
assert out["clean"] == [1.0, 2.0, -1.0]
assert out["k_smallest"] == [-1.0, 1.0]
assert out["with_x"] == [-1.0, 1.0, 1.5, 2.0]
assert out["probe_out"] is None

out2 = stdlib_pack([5.0, 5.0, 1.0], k=10, x=5.0, probe=False)
assert out2["k_smallest"] == [1.0, 5.0, 5.0]
assert out2["with_x"] == [1.0, 5.0, 5.0, 5.0]  # bisect_left inserts before existing 5.0s

try:
    stdlib_pack([float("nan"), float("inf")], k=1, x=0.0, probe=False)
    assert False, "Expected ValueError on empty clean list"
except ValueError:
    pass